# Sensitivity of filter feeders to spawning: amount and frequency

Adult suspension feeders (`Y3`) spawn larvae, which become juveniles (`Yy3`).
In `src/BFM/Ben/FilterFeeder.F90` the fraction of adult biomass spawned in one event is

```
sx = p_xspawning - Ys3c / Y3c        (Ys3c: larvae already spawned)
```

and an event only happens when `sx > p_lxspawning` (and the feeders are active, food
saturation is above `p_fsat_y`, the temperature is above `p_clxTemp` and the bed shear
stress is low). The spawned biomass goes from `Y3` to `Yy3` (flux `jspaY3c`).

The OysterGrounds case uses `p_xspawning = 0.3` and `p_lxspawning = 0.2` in
`FilterFeeder.nml` (`/FilterFeeder_parameters/`). Both are single values (not per group)
and are read from the namelist, so **no recompilation is needed**.

**Lower both together.** With only `p_xspawning` reduced to 0.2 or less, `sx` can never
exceed `p_lxspawning = 0.2`, and spawning stops completely. `p_xspawning = 0` switches
spawning off (`"0/0"`).

**How often spawning happens.** The code comment says about 2 events per year, but the
event flux is capped by `LimitChange_vector` (`p_max_state_change = 0.05` per 300 s step).
An event therefore lasts about 3 steps, moves about 12 % of `Y3c` to `Yy3`, and stops as soon
as `Ys3c/Y3c` passes 0.1. `Ys3c` then only has to decay by about a quarter (rate
`p_smYy3c/et`) before the next event: roughly 3 to 7 weeks, shorter when adults grow or
juveniles are short of food. The default scenarios test the settings that control this:

| scenario | effect |
|---|---|
| `0.30/0.20` | reference (case values) |
| `0/0` | no spawning |
| `0.30/0.25` | new event only when `Ys3c/Y3c` < 0.05 instead of 0.1 |
| `p_smYy3c=0.006`, `0.003` | slower decay of `Ys3c`: longer gaps (also slower juvenile to adult transfer) |
| `p_clxTemp=10`, `14` | spawning only above 10 or 14 °C instead of 6 °C: shorter season |
| `p_fsat_y=0.95`, `0.99` | spawning only at higher food saturation |

Outputs, over one year:

* **(a)** annual mean of `param_var` (default `Y3c`) per scenario, with the % change from
  the reference and the number of spawning events
* **(b)** daily values through the year, one line per scenario
* **(c)** the days with spawning (`jspaY3c` > 0, a daily average) for each scenario
* a table (printed and saved as CSV) with the parameters, the number of events, the days with
  spawning, the carbon spawned, and the annual means of `Y3c`, `Yy3c`, `Y3c+Yy3c` and `jspaY3c`

Spawning only moves carbon from adults to juveniles, so look at `Y3c+Yy3c` for the effect on
the total suspension-feeder biomass. One year takes about 6 minutes per run; the runs are
started in parallel.

## NaaVRE workflow

```
[spawning scenarios] --> [run GOTM spawning] --> [plot spawning]
   param_scenarios: List    run_dirs: List          results_folder: String
```

1. **spawning scenarios**: the list of scenarios, items `"xspawning/lxspawning"` with
   optional `p_smYy3c=..`, `p_clxTemp=..`, `p_fsat_y=..`.
2. **run GOTM spawning**: one model run per scenario, in parallel; outputs `run_dirs`,
   the run folders under `/tmp/data/` (the only folder shared between components).
3. **plot spawning**: reads `OG.nc` from each folder, makes the table and figure and saves
   them, with each run's `log.out` and `run_info.json`, to Cloud Storage, because NaaVRE
   clears `/tmp/data/` when the workflow finishes. It outputs that folder as `results_folder`.

In [ ]:
# spawning scenarios
# ---
# NaaVRE:
#  cell:
#   inputs: []
#   outputs:
#    - param_scenarios: List
# ...
# One item per model run: "p_xspawning/p_lxspawning", optionally followed by
# NAME=VALUE for p_smYy3c, p_clxTemp or p_fsat_y (separated by spaces).
# Case values: 0.30/0.20, p_smYy3c=0.013, p_clxTemp=6, p_fsat_y=0.9 (the reference).
#   "0/0"                 no spawning at all (p_xspawning = 0 switches it off)
#   p_xspawning-p_lxspawning   Ys3c/Y3c must be below this for a new event (0.1)
#   p_smYy3c              decay rate of Ys3c [1/d]: lower -> longer gap between events
#                         (also the juvenile -> adult transfer rate)
#   p_clxTemp             minimum bottom temperature [C]: higher -> shorter season
#   p_fsat_y              minimum food saturation (0-1): higher -> only at the richest food
param_scenarios = [
    "0.30/0.20",                                          # reference (case values)
    "0/0",                                                # no spawning
    "0.30/0.25",                                          # threshold 0.05 instead of 0.1
    "0.30/0.20 p_smYy3c=0.006", "0.30/0.20 p_smYy3c=0.003",
    "0.30/0.20 p_clxTemp=10", "0.30/0.20 p_clxTemp=14",
    "0.30/0.20 p_fsat_y=0.95", "0.30/0.20 p_fsat_y=0.99",
    # smaller events: "0.20/0.133", "0.10/0.067"
]
print("param_scenarios:", param_scenarios)

# Empty the run folder once, here, before the splitter: the parallel branches of the
# run cell share this folder, so they must not empty it themselves.
# gotm-run copies the case from the read-only image, so its folders are read-only:
# add write permission (like chmod -R u+w) before deleting.
import os
import shutil
import stat

results_root = "/tmp/data/Spawning"   # same as param_results_root in the run cell


def _make_writable(func, path, exc_info):
    for p in (os.path.dirname(path), path):
        try:
            os.chmod(p, os.stat(p).st_mode | stat.S_IRWXU)
        except OSError:
            pass
    func(path)


if os.path.isdir(results_root) and os.listdir(results_root):
    print("Emptying", results_root)
    for name in os.listdir(results_root):
        path = os.path.join(results_root, name)
        if os.path.isdir(path) and not os.path.islink(path):
            shutil.rmtree(path, onerror=_make_writable)
        else:
            _make_writable(os.remove, path, None)
os.makedirs(results_root, exist_ok=True)


In [ ]:
# run GOTM spawning
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - param_scenarios: List
#   outputs:
#    - run_dirs: List
#   params:
#    - param_year:
#       type: Integer
#       default_value: 1972
#    - param_var:
#       type: String
#       default_value: "Y3c"
#    - param_start:
#       type: String
#       default_value: "1972-01-01 00:00:00"
#    - param_stop:
#       type: String
#       default_value: "1973-01-01 00:00:00"
# ...


# Settings
param_start = "1972-01-01 00:00:00"
param_stop = "1973-01-01 00:00:00"
param_results_root = "/tmp/data/Spawning"   # shared between workflow components
param_year = 1972            # year to average over
param_var = "Y3c"            # BFM variable in OG.nc to plot, e.g. "Yy3c" or "jspaY3c"
param_cloud_folder = "/home/jovyan/Cloud Storage/naa-vre-user-data/Spawning"   # kept after the workflow

# Spawning scenarios: labels "xspa_<x>_lxspa_<y>_sm_<s>_clxT_<t>_fsat_<f>", one per run.
# param_scenarios (List, from the spawning scenarios cell): items
# "xspawning/lxspawning [p_smYy3c=..] [p_clxTemp=..] [p_fsat_y=..]"; a single number x
# sets p_lxspawning to 2/3 * x. Missing values are the case values.
CASE = {"p_smYy3c": 0.013, "p_clxTemp": 6.0, "p_fsat_y": 0.9}
items = param_scenarios if isinstance(param_scenarios, (list, tuple)) else str(param_scenarios).split(",")
scenarios = []
for item in items:
    if isinstance(item, (list, tuple)):
        tokens = [str(v) for v in item]
    else:
        tokens = str(item).replace(";", " ").split()
    if not tokens:
        continue
    extra = dict(CASE)
    values = []
    for tok in tokens:
        if "=" in tok:
            name, val = tok.split("=", 1)
            if name.strip() not in CASE:
                raise ValueError("unknown parameter {} in {!r}; use {}".format(name, item, ", ".join(CASE)))
            extra[name.strip()] = float(val)
        else:
            values += [float(v) for v in tok.split("/") if v.strip()]
    xspa = values[0] if values else 0.3
    lxspa = values[1] if len(values) > 1 else xspa * 2.0 / 3.0
    if lxspa >= xspa and xspa > 0:
        print("WARNING: p_lxspawning >= p_xspawning in {}: no spawning in this run".format(item))
    scenarios.append("xspa_{:.3f}_lxspa_{:.3f}_sm_{:g}_clxT_{:g}_fsat_{:g}".format(
        xspa, lxspa, extra["p_smYy3c"], extra["p_clxTemp"], extra["p_fsat_y"]))
print("scenarios:", scenarios)

# run GOTM spawning
# Output run_dirs (List of str): run folders under param_results_root.
import os
import re
import shutil
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor
import stat


def remove_tree(path):
    """Delete a file or folder. gotm-run copies the case from the read-only image, so
    its folders are read-only: add write permission (like chmod -R u+w) and retry."""
    def make_writable(func, p, exc_info):
        for q in (os.path.dirname(p), p):
            try:
                os.chmod(q, os.stat(q).st_mode | stat.S_IWUSR | stat.S_IRUSR | stat.S_IXUSR)
            except OSError:
                pass
        func(p)
    if os.path.isdir(path) and not os.path.islink(path):
        shutil.rmtree(path, onerror=make_writable)
    else:
        try:
            os.remove(path)
        except PermissionError:
            make_writable(os.remove, path, None)




# Each branch only deletes its own run folder; the folder is emptied in the
# scenarios cell.
def run_one(label):
    """One GOTM-BFM run for a label "xspa_<x>_lxspa_<y>_sm_<s>_clxT_<t>_fsat_<f>"."""
    parts = label.split("_")
    xspa, lxspa, sm, clxT, fsat = (float(v) for v in parts[1::2])
    run_dir = os.path.join(param_results_root, label)
    if os.path.exists(run_dir):          # gotm-run needs a new or empty directory
        remove_tree(run_dir)
    os.makedirs(param_results_root, exist_ok=True)
    command = [
        GOTM_RUN,
        "--out", run_dir,
        "--start", param_start,
        "--stop", param_stop,
        "--set", "FilterFeeder.nml:p_xspawning={:g}".format(xspa),
        "--set", "FilterFeeder.nml:p_lxspawning={:g}".format(lxspa),
        "--set", "FilterFeeder.nml:p_smYy3c={:g}".format(sm),
        "--set", "FilterFeeder.nml:p_clxTemp={:g}".format(clxT),
        "--set", "FilterFeeder.nml:p_fsat_y={:g}".format(fsat),
    ]
    proc = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    status = "finished" if proc.returncode == 0 else "FAILED (see {}/log.out)".format(run_dir)
    # namelist echo in log.out: the values the model actually used
    used = ""
    log_file = os.path.join(run_dir, "log.out")
    if os.path.exists(log_file):
        with open(log_file, errors="replace") as f:
            used = " | ".join(" ".join(u.split()) for u in
                              re.findall(r"P_(?:L?XSPAWNING|SMYY3C|CLXTEMP|FSAT_Y)\s*=[-+.0-9Ee, ]*", f.read(), flags=re.IGNORECASE))
    print("{:<44} {}   {}".format(label, status, used or "(values not found in log.out)"), flush=True)
    if proc.returncode != 0:
        print(proc.stdout.decode(errors="replace")[-2000:])
    return run_dir


# gotm-run is installed in the GOTM-BFM image (/usr/local/bin/gotm-run); stop here,
# before anything is deleted, if this kernel cannot find it
GOTM_RUN = shutil.which("gotm-run") or (
    "/usr/local/bin/gotm-run" if os.access("/usr/local/bin/gotm-run", os.X_OK) else None)
if GOTM_RUN is None:
    raise RuntimeError(
        "gotm-run not found. Run this notebook in the virtual lab built from the GOTM-BFM "
        "Dockerfile, not in this kernel ({}).".format(sys.prefix))
print("model:", GOTM_RUN)

print("Starting {} run(s) from {} to {} ...".format(len(scenarios), param_start, param_stop), flush=True)
with ThreadPoolExecutor(max_workers=max(1, len(scenarios))) as pool:
    run_dirs = list(pool.map(run_one, scenarios))
print("run_dirs:", run_dirs)

In [ ]:
# plot spawning
# ---
# NaaVRE:
#  cell:
#   inputs:
#    - run_dirs: List
#   outputs:
#    - results_folder: String
#   params:
#    - param_year:
#       type: Integer
#       default_value: 1972
#    - param_var:
#       type: String
#       default_value: "Y3c"
#    - param_cloud_folder:
#       type: String
#       default_value: "/home/jovyan/Cloud Storage/naa-vre-user-data/Spawning"
# ...

# plot spawning
# Input run_dirs (List of str): run folders from the run cell.
# Output results_folder (str): Cloud Storage folder with the figure and the table.
import datetime
import json
import os
import shutil

import matplotlib.pyplot as plt
import netCDF4
import numpy as np
import stat


def remove_tree(path):
    """Delete a file or folder. gotm-run copies the case from the read-only image, so
    its folders are read-only: add write permission (like chmod -R u+w) and retry."""
    def make_writable(func, p, exc_info):
        for q in (os.path.dirname(p), p):
            try:
                os.chmod(q, os.stat(q).st_mode | stat.S_IWUSR | stat.S_IRUSR | stat.S_IXUSR)
            except OSError:
                pass
        func(p)
    if os.path.isdir(path) and not os.path.islink(path):
        shutil.rmtree(path, onerror=make_writable)
    else:
        try:
            os.remove(path)
        except PermissionError:
            make_writable(os.remove, path, None)



# Flatten, in case run_dirs arrives as a list of lists
folders = []
for item in (run_dirs if isinstance(run_dirs, (list, tuple)) else [run_dirs]):
    folders.extend(item if isinstance(item, (list, tuple)) else [item])

year = int(param_year)
var = str(param_var)
table_vars = [var] + [v for v in ("Y3c", "Yy3c", "jspaY3c") if v != var]
NAMES = ["p_xspawning", "p_lxspawning", "p_smYy3c", "p_clxTemp", "p_fsat_y"]
CASE = [0.3, 0.2, 0.013, 6.0, 0.9]
close = lambda a, b: abs(a - b) < 1e-6

results = []    # (params, dates, values of var, {table var: annual mean}, event info)
units, long_name = "", var
for run_dir in folders:
    label = os.path.basename(os.path.normpath(run_dir))
    info_file = os.path.join(run_dir, "run_info.json")
    nc_file = os.path.join(run_dir, "OG.nc")
    if os.path.exists(info_file):
        with open(info_file) as f:
            if json.load(f).get("status") != "finished":
                print("skipped (run failed):", run_dir)
                continue
    if not os.path.exists(nc_file):
        print("skipped (no OG.nc):", run_dir)
        continue
    # xspa_<x>_lxspa_<y>[_sm_<s>_clxT_<t>_fsat_<f>]; older labels have only the first two
    values = [float(v) for v in label.split("_")[1::2]]
    params = values + CASE[len(values):]
    with netCDF4.Dataset(nc_file) as nc:
        t = nc.variables["time"]
        # num2date may return cftime dates, which matplotlib cannot plot:
        # convert them to standard Python datetimes
        dates = [datetime.datetime(d.year, d.month, d.day, d.hour, d.minute, d.second)
                 for d in netCDF4.num2date(t[:], t.units)]
        keep = np.array([d.year == year for d in dates])
        series = {}
        for v in table_vars:
            if v in nc.variables:
                series[v] = np.ma.filled(nc.variables[v][:].reshape(len(dates), -1)[:, 0], np.nan)[keep]
        units = getattr(nc.variables[var], "units", "")
        long_name = getattr(nc.variables[var], "long_name", var)
    dates = [d for d, k in zip(dates, keep) if k]
    annual = {v: np.nanmean(s) for v, s in series.items()}
    if "Y3c" in series and "Yy3c" in series:          # adults + juveniles
        annual["Y3c+Yy3c"] = np.nanmean(series["Y3c"] + series["Yy3c"])
    # Spawning events from jspaY3c (saved as an average per output step, one day):
    # an event is a run of consecutive output steps with jspaY3c > 0.
    events = {"n": 0, "days": 0, "dates": [], "spawned": np.nan}
    if "jspaY3c" in series:
        jspa = np.nan_to_num(series["jspaY3c"])
        on = jspa > 1e-6
        starts = np.flatnonzero(on & ~np.r_[False, on[:-1]])
        step = (dates[1] - dates[0]).total_seconds() / 86400.0 if len(dates) > 1 else 1.0
        events = {"n": len(starts), "days": int(on.sum()) * step,
                  "dates": [dates[i] for i in np.flatnonzero(on)],
                  "spawned": float(np.sum(jspa) * step)}     # mg C/m2 over the year
    results.append((params, dates, series[var], annual, events))
if not results:
    raise RuntimeError("no finished runs in {}".format(folders))


def short_label(p):
    """Name a scenario by what differs from the case values."""
    if p[0] == 0:
        return "no spawning"
    diff = ["{} {:g}".format(n.replace("p_", ""), v) for n, v, c in zip(NAMES, p, CASE)
            if not close(v, c)]
    return "\n".join(diff) if diff else "reference"


results.sort(key=lambda r: (short_label(r[0]) != "reference", short_label(r[0]) != "no spawning",
                            short_label(r[0])))
labels = [short_label(r[0]) for r in results]
i_ref = next((i for i, l in enumerate(labels) if l == "reference"), 0)
mean_of = lambda r, v: r[3].get(v, np.nan)
means = np.array([mean_of(r, var) for r in results])
change = 100.0 * (means - means[i_ref]) / means[i_ref]

# Table: parameters, events, and annual means with % change from the reference
cols = [v for v in table_vars + ["Y3c+Yy3c"] if any(v in r[3] for r in results)]
header = list(NAMES) + ["n_events", "days_spawning", "spawned_mgC_m2"]
for v in cols:
    header += ["mean_" + v, "change_" + v + "_percent"]
table = [",".join(header)]
print("reference:", labels[i_ref].replace("\n", ", "))
widths = [max(len(h), 8) for h in header]
print("  ".join(h.rjust(w) for h, w in zip(header, widths)))
for r in results:
    row = list(r[0]) + [r[4]["n"], r[4]["days"], r[4]["spawned"]]
    for v in cols:
        m, m_ref = mean_of(r, v), mean_of(results[i_ref], v)
        row += [m, 100.0 * (m - m_ref) / m_ref if m_ref else np.nan]
    print("  ".join("{:.4g}".format(x).rjust(w) for x, w in zip(row, widths)))
    table.append(",".join("{:.6g}".format(x) for x in row))

# Plot: (a) annual mean, (b) daily values, (c) spawning days per scenario
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5),
                                    gridspec_kw={"width_ratios": [1.1, 1.2, 1.0]})
colors = plt.cm.viridis(np.linspace(0, 0.9, len(results)))
colors[i_ref] = (0, 0, 0, 1)
x = np.arange(len(results))

ax1.bar(x, means, color=colors, width=0.6)
for xi, m, c, r in zip(x, means, change, results):
    ax1.annotate("{:+.1f}%\n{} ev.".format(c, r[4]["n"]), (xi, m), textcoords="offset points",
                 xytext=(0, 4), ha="center", fontsize=8)
ax1.set_xticks(x)
ax1.set_xticklabels(labels, fontsize=8)
ax1.set_ylabel("annual mean {} [{}]".format(var, units))
ax1.set_title("(a) {}, mean over {} (% change, events)".format(long_name, year), fontsize=10)
ax1.margins(y=0.18)

for i, (r, col, l) in enumerate(zip(results, colors, labels)):
    ax2.plot(r[1], r[2], color=col, lw=1.6 if i == i_ref else 1.0,
             label=l.replace("\n", ", "))
ax2.set_ylabel("{} [{}]".format(var, units))
ax2.set_title("(b) daily values in {}".format(year), fontsize=10)
ax2.legend(fontsize=7, frameon=False)

for xi, (r, col) in enumerate(zip(results, colors)):
    if r[4]["dates"]:
        ax3.plot(r[4]["dates"], [xi] * len(r[4]["dates"]), "|", color=col, ms=12, mew=2)
ax3.set_yticks(x)
ax3.set_yticklabels([l.replace("\n", ", ") for l in labels], fontsize=8)
ax3.invert_yaxis()
ax3.set_xlim(datetime.datetime(year, 1, 1), datetime.datetime(year + 1, 1, 1))
ax3.set_title("(c) days with spawning (jspaY3c > 0)", fontsize=10)
ax3.grid(axis="x", color="0.9")
fig.autofmt_xdate()
plt.setp(ax1.get_xticklabels(), rotation=35, ha="right", rotation_mode="anchor")
fig.tight_layout()

# Save to Cloud Storage: /tmp/data is cleared when the workflow finishes
results_folder = str(param_cloud_folder)
# Empty the Cloud Storage folder first, so files from an earlier submission are not mixed in
if os.path.isdir(results_folder) and os.listdir(results_folder):
    print("Emptying", results_folder)
    for name in os.listdir(results_folder):
        remove_tree(os.path.join(results_folder, name))
os.makedirs(results_folder, exist_ok=True)
figure_file = os.path.join(results_folder, "Spawning_{}_{}.png".format(var, year))
fig.savefig(figure_file, dpi=150)
with open(os.path.join(results_folder, "Spawning_{}.csv".format(year)), "w") as f:
    f.write("\n".join(table) + "\n")
for run_dir in folders:                  # keep the model logs and run info as well
    name = os.path.basename(os.path.normpath(run_dir))
    for fname in ("log.out", "run_info.json"):
        if os.path.exists(os.path.join(run_dir, fname)):
            os.makedirs(os.path.join(results_folder, name), exist_ok=True)
            shutil.copy(os.path.join(run_dir, fname), os.path.join(results_folder, name, fname))
print("results_folder:", results_folder)
print("figure:", figure_file)
plt.show()